# Quantifying Mtb on a single-cell level across large tissue slices

First work on a single example to determine thresholds then iterate over all slices.

In [ ]:
import glob
import os

import dask.array as da
import napari
from tqdm.auto import tqdm


In [2]:
# Define the keywords for the preferred Zarr files
KEYWORDS = ('top', 'bot', 'left', 'right')

# The initial state: all Zarr file paths
all_zarr_paths = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*zarr')

# Group paths by their parent directory (mouse_N/zarr)
paths_by_parent = {}
for path in all_zarr_paths:
    parent_dir = os.path.dirname(path)
    if parent_dir not in paths_by_parent:
        paths_by_parent[parent_dir] = []
    paths_by_parent[parent_dir].append(path)

# Filter the paths
filtered_addresses = []
for paths in paths_by_parent.values():
    # Identify Zarr files that contain any of the keywords
    keyword_zarrs = [
        path for path in paths
        if any(keyword in os.path.basename(path) for keyword in KEYWORDS)
    ]

    # If keyword Zarrs exist, use only them; otherwise, use all paths in the group
    if keyword_zarrs:
        filtered_addresses.extend(keyword_zarrs)
    else:
        filtered_addresses.extend(paths)

# The streamlined result
print_list = [print(fn + '\n') for fn in filtered_addresses]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5573.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/zarr/rep2_mouse3_top.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_3/zarr/rep2_mouse3_bot.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/zarr/rep2_mouse4_bot.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_4/zarr/rep2_mouse4_top.zarr

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_5/zarr/rep2_mouse5_bot.zarr

/mnt/OPERA3/Nathan/data/macroh

In [4]:
viewer = napari.Viewer(title = 'testing cellpose params')

In [6]:
for zarr_address in tqdm(filtered_addresses):
    # Find which, if any, keyword applies to this specific Zarr address
    split = next(
        (keyword for keyword in KEYWORDS if keyword in os.path.basename(zarr_address)),
        None  # Use None if no keyword is found
    )
    # --- 1. Load Data Conditionally and Determine Output Path ---
    if split:
        # Load s0 level of max projected images for a SPLIT Zarr
        images = da.from_zarr(f"{zarr_address}/s0")
        # Set the output subdirectory name based on the split keyword
        output_sub_dir = f"dapi_segmentation_{split}"
        masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_{split}/0")
    else:
        # Load max projected images for an ORIGINAL FULL IMAGE Zarr
        images = da.from_zarr(f"{zarr_address}/0/0")[0].max(axis=1)
        # Set the default output subdirectory name for whole images
        output_sub_dir = "dapi_segmentation"
        # masks = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")

    dapi_channel = images[0].compute()
    # masks = masks.compute()

    break
    # dapi_channel_norm = normalize(dapi_channel)


  0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
zarr_

In [7]:
masks = da.from_zarr(f"{zarr_address}/labels/dapi_segmentation/0")

In [8]:
masks = masks.compute()

In [14]:
# viewer.add_image(dapi_channel)
viewer.add_labels(masks)

image.py (274): data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
image.py (274): data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Labels layer 'masks [1]' at 0x745b0e3712b0>

In [9]:
%%time
dapi_channel_norm = normalize(dapi_channel)


NameError: name 'normalize' is not defined

In [ ]:
PREDICTION_TILES = (64, 72)

### 

In [23]:
%%time
print(f"Running StarDist prediction for {os.path.basename(zarr_address)}...")
labels, details = model.predict_instances(dapi_channel_norm,
                                          n_tiles=PREDICTION_TILES,
                                          prob_thresh=0.25, 
                                          nms_thresh=0.2
                                          )


Running StarDist prediction for 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.zarr...


2025-12-11 11:57:55.482612: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2025-12-11 11:57:55.483044: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 3000000000 Hz
100%|██████████████████████████████████████████████████████████████████████████████| 4672/4672 [27:25<00:00,  2.84it/s]


CPU times: user 8h 8min 49s, sys: 1h 5min 31s, total: 9h 14min 20s
Wall time: 40min 57s


In [24]:
viewer.add_labels(labels)

image.py (274): data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
image.py (274): data shape (45850, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.


<Labels layer 'labels' at 0x745b0e4f5a30>

In [ ]:
print()

In [29]:
execute

NameError: name 'execute' is not defined

In [28]:
print()

In [ ]:
print()